# Seleccion del Mejor Modelo con PyCaret

Este notebook documenta la selección automática del mejor modelo de machine learning utilizando PyCaret sobre el dataset `data/train.csv` con el target `SalePrice`.

---

## 1. Imports y carga de datos

In [ ]:
import pycaret
import pandas as pd
import joblib

data = pd.read_csv('data/train.csv')
data.shape, data.columns

---

## 2. Indentificación del tipo de problema

En base al EDA previo, el target `SalePrice` es de tipo **continuo** (regresión):

In [ ]:
print(data['SalePrice'].describe())
print('Valores únicos:', data['SalePrice'].nunique())

*Conclusión: SalePrice es un valor numérico continuo, por lo que se trata de un problema de **regresión**.*

---

## 3. Import y creación del experimento de PyCaret

In [ ]:
from pycaret.regression import RegressionExperiment

Experimento = RegressionExperiment()

---

## 4. Setup del experimento
Se realiza el setup con PyCaret, indicando el target `SalePrice` y un `session_id` fijo para reproducibilidad. PyCaret utilizará validación cruzada KFold internamente para evaluar de forma justa los modelos.

In [ ]:
Experimento.setup(data, target='SalePrice', session_id=123)

---

## 5. Comparación de modelos
A continuación, PyCaret compara automáticamente más de 20 algoritmos de regresión distintos empleando validación cruzada KFold. Esto garantiza una evaluación robusta y justa para seleccionar el mejor modelo inicial.

In [ ]:
MejorModelo = Experimento.compare_models()
print('El mejor modelo encontrado por PyCaret es:')
print(MejorModelo)

---

## 6. Comparación específica: Top 3 modelos por MSE

Seleccionamos los 3 mejores modelos ordenados por la métrica principal para regresión (MSE).

In [ ]:
Mejores_TOP3 = Experimento.compare_models(sort='MSE', n_select=3)
print('Top 3 modelos (por MSE):')
for i, modelo in enumerate(Mejores_TOP3):
    print(f"Modelo {i+1}:", modelo)

---

## 7. Tuning del mejor modelo

Se realiza un ajuste de hiperparámetros (tuning) sobre el mejor modelo seleccionado para potenciar su desempeño. Usamos sólo 10 iteraciones (`n_iter=10`) para agilizar el ejemplo.

In [ ]:
MejorModelo_Tuneado = Experimento.tune_model(estimator=MejorModelo, n_iter=10)
print('Mejor modelo tuneado:')
print(MejorModelo_Tuneado)

La optimización de hiperparámetros puede encontrar combinaciones mejores que el modelo por defecto, mejorando las métricas de validación cruzada sobre el set de entrenamiento.

---

## 8. Finalización del modelo

El modelo tuneado final se "finaliza" para que esté listo para producción (incluye todos los transformadores y preprocesados en un solo pipeline).

In [ ]:
Modelo_Final = Experimento.finalize_model(estimator=MejorModelo_Tuneado)
print('Modelo final listo para producción:')
print(Modelo_Final)

---

## 9. Guardado del modelo (dos formatos)

Guardamos el pipeline completo de PyCaret y también sólo el estimador tuneado con joblib como referencia extra.

In [ ]:
Experimento.save_model(model=Modelo_Final, model_name='proyecto_pipeline_machine_learning_muestra/Modelo_Final_PyCaret')
joblib.dump(MejorModelo_Tuneado, 'proyecto_pipeline_machine_learning_muestra/modelo_entrenado_solo.pkl')

---

## 10. Carga del modelo guardado (verificación)

Cargamos el pipeline guardado para asegurarnos que la serialización/deserialización funciona correctamente y poder usarlo en despliegues/posterior predicción.

In [ ]:
Modelo_Cargado = Experimento.load_model('proyecto_pipeline_machine_learning_muestra/Modelo_Final_PyCaret')
print('Modelo cargado exitosamente:')
print(Modelo_Cargado)

---

# Conclusión

Este pipeline automatizado permite comparar decenas de algoritmos, seleccionar el mejor según validación cruzada, ajustar hiperparámetros, finalizar y guardar el modelo en forma reproducible y robusta para despliegue.

Para nuevos datos, simplemente cargar el pipeline con `load_model` y llamar `.predict()`.